# MemoryWatch — Deliverable 2, Notebook 3: Model Training

Trains the **Isolation Forest** (Liu, Ting & Zhou, ICDM 2008), reimplemented from scratch in vectorised NumPy since this environment has no sklearn/scipy. Key points:

- Trained on **normal-only** samples (unsupervised anomaly detection, not classification)
- 100 trees, sub-sample size ψ = 256, `random_state = 42` for reproducibility
- Anomaly threshold is **dynamic**: the 99th percentile of the model's own scores on the normal training data, not a fixed global cutoff

Loads `memorywatch_processed.npz` from Notebook 2; saves scores + threshold to `memorywatch_model_scores.npz` for Notebook 4.

In [1]:
import sys
sys.path.insert(0, '.')
import time
import numpy as np
import utils

d = np.load(utils.OUT_DIR + 'memorywatch_processed.npz', allow_pickle=True)
X_tr, y_tr = d['X_tr'], d['y_tr']
X_te, y_te, at_te = d['X_te'], d['y_te'], d['at_te']
X_norm = X_tr[y_tr == 0]
print(f"Training on {len(X_norm):,} normal samples")


Training on 56,000 normal samples


In [2]:
t0 = time.time()
clf = utils.IsolationForest(n_estimators=100, max_samples=256, seed=42)
clf.fit(X_norm)
print(f"Trained 100 trees in {time.time()-t0:.2f}s")


    trees: 25/100
    trees: 50/100
    trees: 75/100
    trees: 100/100
Trained 100 trees in 0.18s


In [3]:
s_norm_train = clf.anomaly_scores(X_norm)
threshold = np.percentile(s_norm_train, 99)
print(f"Dynamic threshold (99th pct of normal scores): {threshold:.6f}")

t0 = time.time()
s_test = clf.anomaly_scores(X_te)
print(f"Scored {len(X_te):,} test samples in {time.time()-t0:.2f}s")


Dynamic threshold (99th pct of normal scores): 0.591159
Scored 82,332 test samples in 0.52s


In [4]:
np.savez(utils.OUT_DIR + 'memorywatch_model_scores.npz',
         s_test=s_test, y_te=y_te, at_te=at_te, threshold=threshold)
print('Saved memorywatch_model_scores.npz')


Saved memorywatch_model_scores.npz
